# Parsing Highest-Grossing Films from Wikipedia

В этом ноутбуке мы извлекаем данные о фильмах с Wikipedia (страница Highest-Grossing Films), очищаем их,  
и подготавливаем JSON для фронтенда и базы данных PostgreSQL.  

Этапы:
1. Импорты библиотек.
2. Определение функций для безопасного запроса, очистки текста и чисел.
3. Парсинг таблицы фильмов.
4. Извлечение деталей каждого фильма (директор, страна).
5. Сохранение результата в JSON.

In [ ]:
"""1"""
from bs4 import BeautifulSoup
import requests, re, time, random
import json

In [ ]:
"""2"""
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

def safe_request(url):
    """Делает запрос с повторной попыткой при ошибке."""
    for _ in range(3):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                return response
        except:
            pass
        time.sleep(1.1)
    return None

def clean_text(txt):
    """Удаляет сноски и лишние символы."""
    txt = re.sub(r"\[.*?\]", "", txt)
    txt = re.sub(r"†", "", txt)
    return txt.strip()

def clean_money(value):
    """Преобразует строку вида $123,456,789 в число."""
    match = re.search(r"\$[\d,]+", value)
    if match:
        number = match.group()
        return float(number.replace("$", "").replace(",", ""))
    return None

def clean_year(value):
    """Извлекает год из текста."""
    value = re.sub(r"[^\d]", "", value)
    return int(value) if value else None

def get_film_details(film_url):
    """Получает режиссера и страну из инфобокса фильма."""
    response = safe_request(film_url)
    if not response:
        return None, None

    soup = BeautifulSoup(response.text, "html.parser")
    infobox = soup.find("table", class_=lambda x: x and "infobox" in x)
    director = None
    country = None

    if infobox:
        rows = infobox.find_all("tr")
        for r in rows:
            header = r.find("th")
            value = r.find("td")
            if not header or not value:
                continue

            header_text = header.get_text(" ", strip=True).lower().replace("\ufeff", "")

            # Режиссер
            if any(k in header_text for k in ["directed by", "director", "directed"]):
                raw_text = clean_text(" ".join(value.stripped_strings))
                parts = []
                if " and " in raw_text:
                    parts = [p.strip() for p in raw_text.split(" and ")]
                elif "," in raw_text:
                    parts = [p.strip() for p in raw_text.split(",")]
                else:
                    parts = [raw_text]

                cleaned = []
                for p in parts:
                    if p and p not in cleaned:
                        cleaned.append(p)
                director = ", ".join(cleaned)

                if not director or len(director) < 3:
                    links = value.find_all("a")
                    names = [clean_text(a.get_text()) for a in links if a.get_text(strip=True)]
                    if names:
                        director = ", ".join(dict.fromkeys(names))

            # Страна
            if any(k in header_text for k in ["country", "countries", "country of origin", "production country"]):
                lis = value.find_all("li")
                if lis:
                    countries = [clean_text(li.get_text()) for li in lis if li.get_text(strip=True)]
                    country = ", ".join(dict.fromkeys(countries))
                else:
                    text = value.get_text(" ", strip=True)
                    text = clean_text(text)
                    if text:
                        country = text
    return director, country

In [ ]:
"""3 and 4"""
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
response = safe_request(url)
soup = BeautifulSoup(response.text, "html.parser")

# Поиск нужной таблицы
tables = soup.find_all("table", {"class": "wikitable"})
table = None
for t in tables:
    headers = [th.get_text(strip=True) for th in t.find_all("th")]
    if "Title" in headers and "Worldwide gross" in headers:
        table = t
        break

rows = table.find_all("tr")

seen = set()
films = []

for row in rows[1:]:
    cols = row.find_all(["td", "th"])
    if len(cols) < 5:
        continue

    title = clean_text(cols[2].get_text(" ", strip=True))
    year = clean_year(cols[4].get_text())
    box_office = clean_money(cols[3].get_text())
    key = (title, year)
    if key in seen:
        continue
    seen.add(key)

    link_tag = cols[2].find("a")
    film_url = None
    if link_tag and link_tag.get("href"):
        film_url = "https://en.wikipedia.org" + link_tag.get("href")

    director, country = None, None
    if film_url:
        director, country = get_film_details(film_url)
        time.sleep(random.uniform(1.1, 1.2))

    films.append({
        "title": title,
        "year": year,
        "box_office": box_office,
        "director": director,
        "country": country
    })

# 5 экспорт:
# Export Films from PostgreSQL to JSON

В этом ноутбуке мы подключаемся к базе PostgreSQL, извлекаем все фильмы из таблицы `films` и экспортируем их в JSON,  
чтобы использовать для фронтенда веб-страницы.

In [ ]:
import json
from pathlib import Path
import psycopg2
from psycopg2.extras import RealDictCursor

In [ ]:
# Настройки подключения к базе
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "films_db"
DB_USER = "postgres"
DB_PASSWORD = "postgres"

# Путь к JSON файлу
OUTPUT_FILE = Path("data/films.json")

In [ ]:
def create_connection():
    """Создает соединение с PostgreSQL"""
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn

def fetch_films(conn):
    """Извлекает фильмы из базы"""
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute("""
            SELECT title, release_year, director, box_office, country
            FROM films
            ORDER BY release_year DESC;
        """)
        return cur.fetchall()

def export_to_json(films, output_file=OUTPUT_FILE):
    """Экспортирует список фильмов в JSON"""
    output_file.parent.mkdir(exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(films, f, ensure_ascii=False, indent=4)
    print(f"Экспортировано {len(films)} фильмов в {output_file}")

In [ ]:
# Создаем соединение
conn = create_connection()

# Получаем фильмы из базы
films = fetch_films(conn)

# Закрываем соединение
conn.close()

# Экспортируем в JSON
export_to_json(films)